In [9]:
"""
SEC HTML Downloader  v2.0
=========================
Downloads 10-K, 10-Q, and 10-Q/A filings as clean, readable HTML files.
Handles iXBRL inline-tagged documents (modern SEC filings) by stripping
XBRL metadata wrappers so the financial content is fully readable.

Folder structure:
    filings/<TICKER>/<YEAR>/<FORM>_<PERIOD>_<ACCNO>.html

No scraping, no LLM, no XBRL parsing — just clean financial HTML on disk.
Covers the last YEARS_BACK calendar years.

Notebook usage:
    tickers = ALL_TICKERS[:100]   # adjust slice
    download_all(tickers)
"""

import os
import re
import sys
import time
import json
import random
import threading
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ─── CONFIG ───────────────────────────────────────────────────────────────────

OUTPUT_DIR          = "filings"   # root: filings/TICKER/YEAR/filename.html
YEARS_BACK          = 7           # calendar years back from today
EDGAR_SLEEP         = 0.15        # seconds between EDGAR requests (be polite)
MAX_RETRIES         = 5
BACKOFF_BASE        = 2.0
BACKOFF_MAX         = 120.0
MIN_HTML_BYTES      = 10_000      # skip suspiciously small files
MAX_HTML_MB         = 80          # skip absurdly large files (MB)
RESUME              = True        # skip files already saved to disk
CLEAN_IXBRL         = True        # strip XBRL wrapper tags → readable HTML

TARGET_FORMS = {"10-K", "10-Q", "10-Q/A", "10-K/A"}

# ─── USER-AGENT POOL ─────────────────────────────────────────────────────────

_USER_AGENTS = [
    "SECDatasetBuilder/2.0 (ML research; research@university.edu)",
    "SECDatasetBuilder/2.1 (Academic use; ml-research@lab.edu)",
    "FinancialDataCollector/2.0 (Research; data@finresearch.org)",
    "SECFilingParser/2.0 (Non-commercial; nlp-lab@university.edu)",
    "AcademicSECBot/2.0 (Research dataset; sec-research@ml.edu)",
]
_ua_idx  = 0
_ua_lock = threading.Lock()

def _next_ua() -> str:
    global _ua_idx
    with _ua_lock:
        ua = _USER_AGENTS[_ua_idx % len(_USER_AGENTS)]
        _ua_idx += 1
    return ua


# ─── iXBRL CLEANER ───────────────────────────────────────────────────────────
# Modern SEC filings (2020+) are Inline XBRL (iXBRL). Browsers render them
# fine, but raw HTML saved to disk looks empty because:
#   1. <ix:hidden> wraps metadata in a display:none div
#   2. Financial values are wrapped in <ix:nonFraction> tags that some
#      parsers/viewers don't understand
# We strip all XBRL wrapper tags while keeping all visible content.

_RE_IX_HIDDEN   = re.compile(r'<ix:hidden[^>]*>.*?</ix:hidden\s*>',          re.DOTALL | re.IGNORECASE)
_RE_IX_NONFRAC  = re.compile(r'<ix:nonFraction[^>]*>(.*?)</ix:nonFraction\s*>', re.DOTALL | re.IGNORECASE)
_RE_IX_NONNUM   = re.compile(r'<ix:nonNumeric[^>]*>(.*?)</ix:nonNumeric\s*>',   re.DOTALL | re.IGNORECASE)
_RE_IX_CONT     = re.compile(r'<ix:continuation[^>]*>(.*?)</ix:continuation\s*>',re.DOTALL | re.IGNORECASE)
_RE_IX_EXCLUDE  = re.compile(r'<ix:exclude[^>]*>(.*?)</ix:exclude\s*>',          re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS     = re.compile(r'</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>',    re.IGNORECASE)
_RE_XBRL_TAGS   = re.compile(r'</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>',re.IGNORECASE)
_RE_LINK_TAGS   = re.compile(r'</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>',  re.IGNORECASE)
_RE_HIDDEN_DIV  = re.compile(r'<div\s[^>]*display\s*:\s*none[^>]*>.*?</div>',re.DOTALL | re.IGNORECASE)

# Inject a minimal CSS reset so the cleaned file renders nicely in a browser
_INJECT_CSS = """
<style>
  body { font-family: Arial, sans-serif; font-size: 10pt; margin: 2em; color: #111; }
  table { border-collapse: collapse; width: 100%; margin: 1em 0; }
  td, th { padding: 3px 8px; border: 1px solid #ccc; vertical-align: top; }
  th { background: #f0f0f0; font-weight: bold; }
  p  { margin: 0.4em 0; line-height: 1.5; }
</style>
"""

def clean_ixbrl(html: str) -> str:
    """
    Strip iXBRL/XBRL wrapper tags from an inline XBRL document.
    Financial values and all visible text/tables are preserved.
    Returns plain HTML that any browser or parser can read.
    """
    # 1. Nuke the ix:hidden block (pure metadata, not shown to users)
    html = _RE_IX_HIDDEN.sub('', html)

    # 2. Unwrap value-holding tags — keep inner content
    html = _RE_IX_NONFRAC.sub(r'\1', html)
    html = _RE_IX_NONNUM.sub(r'\1',  html)
    html = _RE_IX_CONT.sub(r'\1',    html)
    html = _RE_IX_EXCLUDE.sub('',    html)   # ix:exclude = intentionally hidden

    # 3. Remove remaining ix: / xbrl: / link: open+close tags
    html = _RE_IX_TAGS.sub('',   html)
    html = _RE_XBRL_TAGS.sub('', html)
    html = _RE_LINK_TAGS.sub('', html)

    # 4. Remove top-level display:none divs (ix:header wrapper)
    html = _RE_HIDDEN_DIV.sub('', html)

    # 5. Inject minimal CSS for readability if not already present
    if '<style' not in html[:2000]:
        html = html.replace('</head>', _INJECT_CSS + '</head>', 1)

    # 6. Clean up excessive blank lines
    html = re.sub(r'\n{4,}', '\n\n', html)

    return html


def is_ixbrl(html: str) -> bool:
    """Detect whether the HTML is an iXBRL document."""
    head = html[:2000]
    return ('xmlns:ix=' in head or 'ix:header' in head or
            'inlineXBRL' in head or 'ix:nonFraction' in html[:50000])


# ─── TICKER LIST ─────────────────────────────────────────────────────────────

ALL_TICKERS = [
    # Technology
    {"ticker": "AAPL",  "name": "Apple Inc"},
    {"ticker": "MSFT",  "name": "Microsoft Corp"},
    {"ticker": "NVDA",  "name": "NVIDIA Corp"},
    {"ticker": "AVGO",  "name": "Broadcom Inc"},
    {"ticker": "ORCL",  "name": "Oracle Corp"},
    {"ticker": "ADBE",  "name": "Adobe Inc"},
    {"ticker": "CRM",   "name": "Salesforce Inc"},
    {"ticker": "AMD",   "name": "Advanced Micro Devices Inc"},
    {"ticker": "INTC",  "name": "Intel Corp"},
    {"ticker": "QCOM",  "name": "Qualcomm Inc"},
    {"ticker": "TXN",   "name": "Texas Instruments Inc"},
    {"ticker": "IBM",   "name": "International Business Machines Corp"},
    {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
    {"ticker": "NOW",   "name": "ServiceNow Inc"},
    {"ticker": "PANW",  "name": "Palo Alto Networks Inc"},
    {"ticker": "INTU",  "name": "Intuit Inc"},
    {"ticker": "SNOW",  "name": "Snowflake Inc"},
    {"ticker": "CRWD",  "name": "CrowdStrike Holdings Inc"},
    {"ticker": "DELL",  "name": "Dell Technologies Inc"},
    {"ticker": "HPQ",   "name": "HP Inc"},
    {"ticker": "ADI",   "name": "Analog Devices Inc"},
    {"ticker": "MU",    "name": "Micron Technology Inc"},
    {"ticker": "LRCX",  "name": "Lam Research Corp"},
    {"ticker": "KLAC",  "name": "KLA Corp"},
    {"ticker": "AMAT",  "name": "Applied Materials Inc"},
    {"ticker": "MCHP",  "name": "Microchip Technology Inc"},
    {"ticker": "ON",    "name": "ON Semiconductor Corp"},
    {"ticker": "FTNT",  "name": "Fortinet Inc"},
    {"ticker": "NET",   "name": "Cloudflare Inc"},
    {"ticker": "ZS",    "name": "Zscaler Inc"},
    {"ticker": "DDOG",  "name": "Datadog Inc"},
    {"ticker": "MDB",   "name": "MongoDB Inc"},
    {"ticker": "TEAM",  "name": "Atlassian Corp"},
    {"ticker": "WDAY",  "name": "Workday Inc"},
    {"ticker": "ADSK",  "name": "Autodesk Inc"},
    {"ticker": "CDNS",  "name": "Cadence Design Systems Inc"},
    {"ticker": "SNPS",  "name": "Synopsys Inc"},
    {"ticker": "ROP",   "name": "Roper Technologies Inc"},
    {"ticker": "TTD",   "name": "Trade Desk Inc"},
    {"ticker": "HUBS",  "name": "HubSpot Inc"},
    {"ticker": "AKAM",  "name": "Akamai Technologies Inc"},
    {"ticker": "VRSN",  "name": "Verisign Inc"},
    {"ticker": "FFIV",  "name": "F5 Inc"},
    {"ticker": "JNPR",  "name": "Juniper Networks Inc"},
    {"ticker": "MSI",   "name": "Motorola Solutions Inc"},
    {"ticker": "HPE",   "name": "Hewlett Packard Enterprise Co"},
    {"ticker": "CDW",   "name": "CDW Corp"},
    {"ticker": "GLW",   "name": "Corning Inc"},
    {"ticker": "APH",   "name": "Amphenol Corp"},
    {"ticker": "TEL",   "name": "TE Connectivity Ltd"},
    {"ticker": "JBL",   "name": "Jabil Inc"},
    {"ticker": "FLEX",  "name": "Flex Ltd"},
    {"ticker": "STX",   "name": "Seagate Technology Holdings PLC"},
    {"ticker": "WDC",   "name": "Western Digital Corp"},
    {"ticker": "NTAP",  "name": "NetApp Inc"},
    {"ticker": "KEYS",  "name": "Keysight Technologies Inc"},
    {"ticker": "MPWR",  "name": "Monolithic Power Systems Inc"},
    {"ticker": "SWKS",  "name": "Skyworks Solutions Inc"},
    {"ticker": "NXPI",  "name": "NXP Semiconductors NV"},
    {"ticker": "ANET",  "name": "Arista Networks Inc"},
    {"ticker": "IT",    "name": "Gartner Inc"},
    {"ticker": "GEN",   "name": "Gen Digital Inc"},
    {"ticker": "EPAM",  "name": "EPAM Systems Inc"},
    {"ticker": "TYL",   "name": "Tyler Technologies Inc"},
    {"ticker": "PTC",   "name": "PTC Inc"},
    {"ticker": "ANSS",  "name": "ANSYS Inc"},
    {"ticker": "SSNC",  "name": "SS&C Technologies Holdings Inc"},
    {"ticker": "ZM",    "name": "Zoom Video Communications Inc"},
    {"ticker": "DOCU",  "name": "DocuSign Inc"},
    {"ticker": "OKTA",  "name": "Okta Inc"},
    {"ticker": "SMCI",  "name": "Super Micro Computer Inc"},
    {"ticker": "PLTR",  "name": "Palantir Technologies Inc"},
    {"ticker": "FSLR",  "name": "First Solar Inc"},
    {"ticker": "ENPH",  "name": "Enphase Energy Inc"},
    {"ticker": "TER",   "name": "Teradyne Inc"},
    {"ticker": "VSH",   "name": "Vishay Intertechnology Inc"},
    {"ticker": "GL",    "name": "Globe Life Inc"},
    {"ticker": "ZBRA",  "name": "Zebra Technologies Corp"},
    # Financials
    {"ticker": "BRK.B", "name": "Berkshire Hathaway Inc"},
    {"ticker": "JPM",   "name": "JPMorgan Chase & Co"},
    {"ticker": "V",     "name": "Visa Inc"},
    {"ticker": "MA",    "name": "Mastercard Inc"},
    {"ticker": "BAC",   "name": "Bank of America Corp"},
    {"ticker": "WFC",   "name": "Wells Fargo & Co"},
    {"ticker": "GS",    "name": "Goldman Sachs Group Inc"},
    {"ticker": "MS",    "name": "Morgan Stanley"},
    {"ticker": "C",     "name": "Citigroup Inc"},
    {"ticker": "AXP",   "name": "American Express Co"},
    {"ticker": "BLK",   "name": "BlackRock Inc"},
    {"ticker": "SCHW",  "name": "Charles Schwab Corp"},
    {"ticker": "SPGI",  "name": "S&P Global Inc"},
    {"ticker": "ICE",   "name": "Intercontinental Exchange Inc"},
    {"ticker": "MCO",   "name": "Moody's Corp"},
    {"ticker": "CME",   "name": "CME Group Inc"},
    {"ticker": "PNC",   "name": "PNC Financial Services Group Inc"},
    {"ticker": "USB",   "name": "US Bancorp"},
    {"ticker": "TFC",   "name": "Truist Financial Corp"},
    {"ticker": "COF",   "name": "Capital One Financial Corp"},
    {"ticker": "BK",    "name": "Bank of New York Mellon Corp"},
    {"ticker": "STT",   "name": "State Street Corp"},
    {"ticker": "FITB",  "name": "Fifth Third Bancorp"},
    {"ticker": "KEY",   "name": "KeyCorp"},
    {"ticker": "CFG",   "name": "Citizens Financial Group Inc"},
    {"ticker": "HBAN",  "name": "Huntington Bancshares Inc"},
    {"ticker": "RF",    "name": "Regions Financial Corp"},
    {"ticker": "SYF",   "name": "Synchrony Financial"},
    {"ticker": "DFS",   "name": "Discover Financial Services"},
    {"ticker": "MTB",   "name": "M&T Bank Corp"},
    {"ticker": "FIS",   "name": "Fidelity National Information Services Inc"},
    {"ticker": "FISV",  "name": "Fiserv Inc"},
    {"ticker": "GPN",   "name": "Global Payments Inc"},
    {"ticker": "PGR",   "name": "Progressive Corp"},
    {"ticker": "ALL",   "name": "Allstate Corp"},
    {"ticker": "TRV",   "name": "Travelers Companies Inc"},
    {"ticker": "AIG",   "name": "American International Group Inc"},
    {"ticker": "MET",   "name": "MetLife Inc"},
    {"ticker": "PRU",   "name": "Prudential Financial Inc"},
    {"ticker": "CB",    "name": "Chubb Ltd"},
    {"ticker": "AFL",   "name": "Aflac Inc"},
    {"ticker": "AJG",   "name": "Arthur J Gallagher & Co"},
    {"ticker": "MMC",   "name": "Marsh & McLennan Companies Inc"},
    {"ticker": "WTW",   "name": "Willis Towers Watson PLC"},
    {"ticker": "BRO",   "name": "Brown & Brown Inc"},
    {"ticker": "CINF",  "name": "Cincinnati Financial Corp"},
    {"ticker": "WRB",   "name": "WR Berkley Corp"},
    {"ticker": "PFG",   "name": "Principal Financial Group Inc"},
    {"ticker": "TROW",  "name": "T Rowe Price Group Inc"},
    {"ticker": "BEN",   "name": "Franklin Resources Inc"},
    {"ticker": "AMP",   "name": "Ameriprise Financial Inc"},
    {"ticker": "RJF",   "name": "Raymond James Financial Inc"},
    {"ticker": "NTRS",  "name": "Northern Trust Corp"},
    {"ticker": "L",     "name": "Loews Corp"},
    {"ticker": "UNM",   "name": "Unum Group"},
    {"ticker": "AIZ",   "name": "Assurant Inc"},
    {"ticker": "FDS",   "name": "FactSet Research Systems Inc"},
    {"ticker": "MKTX",  "name": "MarketAxess Holdings Inc"},
    {"ticker": "NDAQ",  "name": "Nasdaq Inc"},
    {"ticker": "CBOE",  "name": "Cboe Global Markets Inc"},
    {"ticker": "LPLA",  "name": "LPL Financial Holdings Inc"},
    {"ticker": "IVZ",   "name": "Invesco Ltd"},
    {"ticker": "CMA",   "name": "Comerica Inc"},
    {"ticker": "ZION",  "name": "Zions Bancorporation"},
    {"ticker": "ERIE",  "name": "Erie Indemnity Co"},
    {"ticker": "JKHY",  "name": "Jack Henry & Associates Inc"},
    {"ticker": "BR",    "name": "Broadridge Financial Solutions Inc"},
    {"ticker": "MSCI",  "name": "MSCI Inc"},
    {"ticker": "BLDR",  "name": "Builders FirstSource Inc"},
    {"ticker": "BX",    "name": "Blackstone Inc"},
    # Healthcare
    {"ticker": "LLY",   "name": "Eli Lilly and Co"},
    {"ticker": "JNJ",   "name": "Johnson & Johnson"},
    {"ticker": "UNH",   "name": "UnitedHealth Group Inc"},
    {"ticker": "MRK",   "name": "Merck & Co Inc"},
    {"ticker": "ABBV",  "name": "AbbVie Inc"},
    {"ticker": "PFE",   "name": "Pfizer Inc"},
    {"ticker": "TMO",   "name": "Thermo Fisher Scientific Inc"},
    {"ticker": "ABT",   "name": "Abbott Laboratories"},
    {"ticker": "DHR",   "name": "Danaher Corp"},
    {"ticker": "REGN",  "name": "Regeneron Pharmaceuticals Inc"},
    {"ticker": "VRTX",  "name": "Vertex Pharmaceuticals Inc"},
    {"ticker": "ISRG",  "name": "Intuitive Surgical Inc"},
    {"ticker": "BSX",   "name": "Boston Scientific Corp"},
    {"ticker": "SYK",   "name": "Stryker Corp"},
    {"ticker": "MDT",   "name": "Medtronic plc"},
    {"ticker": "CI",    "name": "Cigna Group"},
    {"ticker": "BMY",   "name": "Bristol-Myers Squibb Co"},
    {"ticker": "GILD",  "name": "Gilead Sciences Inc"},
    {"ticker": "AMGN",  "name": "Amgen Inc"},
    {"ticker": "CVS",   "name": "CVS Health Corp"},
    {"ticker": "ELV",   "name": "Elevance Health Inc"},
    {"ticker": "HUM",   "name": "Humana Inc"},
    {"ticker": "MRNA",  "name": "Moderna Inc"},
    {"ticker": "BIIB",  "name": "Biogen Inc"},
    {"ticker": "ZTS",   "name": "Zoetis Inc"},
    {"ticker": "IQV",   "name": "IQVIA Holdings Inc"},
    {"ticker": "LH",    "name": "Labcorp Holdings Inc"},
    {"ticker": "DGX",   "name": "Quest Diagnostics Inc"},
    {"ticker": "CAH",   "name": "Cardinal Health Inc"},
    {"ticker": "MCK",   "name": "McKesson Corp"},
    {"ticker": "COR",   "name": "Cencora Inc"},
    {"ticker": "BAX",   "name": "Baxter International Inc"},
    {"ticker": "BDX",   "name": "Becton Dickinson and Co"},
    {"ticker": "GEHC",  "name": "GE HealthCare Technologies Inc"},
    {"ticker": "RMD",   "name": "ResMed Inc"},
    {"ticker": "STE",   "name": "Steris PLC"},
    {"ticker": "HCA",   "name": "HCA Healthcare Inc"},
    {"ticker": "CNC",   "name": "Centene Corp"},
    {"ticker": "HOLX",  "name": "Hologic Inc"},
    {"ticker": "IDXX",  "name": "IDEXX Laboratories Inc"},
    {"ticker": "MTD",   "name": "Mettler-Toledo International Inc"},
    {"ticker": "WST",   "name": "West Pharmaceutical Services Inc"},
    {"ticker": "EW",    "name": "Edwards Lifesciences Corp"},
    {"ticker": "DXCM",  "name": "DexCom Inc"},
    {"ticker": "ALGN",  "name": "Align Technology Inc"},
    {"ticker": "INCY",  "name": "Incyte Corp"},
    {"ticker": "PODD",  "name": "Insulet Corp"},
    {"ticker": "MASI",  "name": "Masimo Corp"},
    {"ticker": "CRL",   "name": "Charles River Laboratories International Inc"},
    {"ticker": "TECH",  "name": "Bio-Techne Corp"},
    {"ticker": "MEDP",  "name": "Medpace Holdings Inc"},
    {"ticker": "BRKR",  "name": "Bruker Corp"},
    {"ticker": "WAT",   "name": "Waters Corp"},
    {"ticker": "A",     "name": "Agilent Technologies Inc"},
    {"ticker": "RVTY",  "name": "Revvity Inc"},
    {"ticker": "COO",   "name": "Cooper Cos Inc"},
    {"ticker": "HWM",   "name": "Howmet Aerospace Inc"},
    {"ticker": "UHS",   "name": "Universal Health Services Inc"},
    {"ticker": "VTRS",  "name": "Viatris Inc"},
    {"ticker": "MOH",   "name": "Molina Healthcare Inc"},
    {"ticker": "DVA",   "name": "DaVita Inc"},
    {"ticker": "XRAY",  "name": "Dentsply Sirona Inc"},
    {"ticker": "HSIC",  "name": "Henry Schein Inc"},
    {"ticker": "SOLV",  "name": "Solventum Corp"},
    {"ticker": "BNTX",  "name": "BioNTech SE"},
    # Consumer Discretionary
    {"ticker": "AMZN",  "name": "Amazon.com Inc"},
    {"ticker": "TSLA",  "name": "Tesla Inc"},
    {"ticker": "HD",    "name": "Home Depot Inc"},
    {"ticker": "LOW",   "name": "Lowe's Companies Inc"},
    {"ticker": "SBUX",  "name": "Starbucks Corp"},
    {"ticker": "MCD",   "name": "McDonald's Corp"},
    {"ticker": "BKNG",  "name": "Booking Holdings Inc"},
    {"ticker": "TJX",   "name": "TJX Companies Inc"},
    {"ticker": "NKE",   "name": "Nike Inc"},
    {"ticker": "ABNB",  "name": "Airbnb Inc"},
    {"ticker": "CMG",   "name": "Chipotle Mexican Grill Inc"},
    {"ticker": "MAR",   "name": "Marriott International Inc"},
    {"ticker": "HLT",   "name": "Hilton Worldwide Holdings Inc"},
    {"ticker": "RCL",   "name": "Royal Caribbean Cruises Ltd"},
    {"ticker": "CCL",   "name": "Carnival Corp"},
    {"ticker": "DHI",   "name": "DR Horton Inc"},
    {"ticker": "LEN",   "name": "Lennar Corp"},
    {"ticker": "NVR",   "name": "NVR Inc"},
    {"ticker": "PHM",   "name": "PulteGroup Inc"},
    {"ticker": "TOL",   "name": "Toll Brothers Inc"},
    {"ticker": "F",     "name": "Ford Motor Co"},
    {"ticker": "GM",    "name": "General Motors Co"},
    {"ticker": "APTV",  "name": "Aptiv PLC"},
    {"ticker": "BWA",   "name": "BorgWarner Inc"},
    {"ticker": "LKQ",   "name": "LKQ Corp"},
    {"ticker": "GPC",   "name": "Genuine Parts Co"},
    {"ticker": "DRI",   "name": "Darden Restaurants Inc"},
    {"ticker": "YUM",   "name": "Yum! Brands Inc"},
    {"ticker": "DPZ",   "name": "Domino's Pizza Inc"},
    {"ticker": "QSR",   "name": "Restaurant Brands International Inc"},
    {"ticker": "TSCO",  "name": "Tractor Supply Co"},
    {"ticker": "ULTA",  "name": "Ulta Beauty Inc"},
    {"ticker": "DG",    "name": "Dollar General Corp"},
    {"ticker": "DLTR",  "name": "Dollar Tree Inc"},
    {"ticker": "ROST",  "name": "Ross Stores Inc"},
    {"ticker": "EBAY",  "name": "eBay Inc"},
    {"ticker": "AZO",   "name": "AutoZone Inc"},
    {"ticker": "ORLY",  "name": "O'Reilly Automotive Inc"},
    {"ticker": "BBY",   "name": "Best Buy Co Inc"},
    {"ticker": "POOL",  "name": "Pool Corp"},
    {"ticker": "HAS",   "name": "Hasbro Inc"},
    {"ticker": "MAT",   "name": "Mattel Inc"},
    {"ticker": "ETSY",  "name": "Etsy Inc"},
    {"ticker": "CVNA",  "name": "Carvana Co"},
    {"ticker": "WYNN",  "name": "Wynn Resorts Ltd"},
    {"ticker": "LVS",   "name": "Las Vegas Sands Corp"},
    {"ticker": "MGM",   "name": "MGM Resorts International"},
    {"ticker": "CZR",   "name": "Caesars Entertainment Inc"},
    {"ticker": "TPR",   "name": "Tapestry Inc"},
    {"ticker": "PVH",   "name": "PVH Corp"},
    {"ticker": "RL",    "name": "Ralph Lauren Corp"},
    {"ticker": "DECK",  "name": "Deckers Outdoor Corp"},
    {"ticker": "GRMN",  "name": "Garmin Ltd"},
    {"ticker": "LULU",  "name": "Lululemon Athletica Inc"},
    # Communication Services
    {"ticker": "META",  "name": "Meta Platforms Inc"},
    {"ticker": "GOOGL", "name": "Alphabet Inc Class A"},
    {"ticker": "GOOG",  "name": "Alphabet Inc Class C"},
    {"ticker": "NFLX",  "name": "Netflix Inc"},
    {"ticker": "DIS",   "name": "Walt Disney Co"},
    {"ticker": "CMCSA", "name": "Comcast Corp"},
    {"ticker": "T",     "name": "AT&T Inc"},
    {"ticker": "VZ",    "name": "Verizon Communications Inc"},
    {"ticker": "TMUS",  "name": "T-Mobile US Inc"},
    {"ticker": "EA",    "name": "Electronic Arts Inc"},
    {"ticker": "TTWO",  "name": "Take-Two Interactive Software Inc"},
    {"ticker": "WBD",   "name": "Warner Bros Discovery Inc"},
    {"ticker": "PARA",  "name": "Paramount Global"},
    {"ticker": "FOXA",  "name": "Fox Corp Class A"},
    {"ticker": "FOX",   "name": "Fox Corp Class B"},
    {"ticker": "NWSA",  "name": "News Corp Class A"},
    {"ticker": "NWS",   "name": "News Corp Class B"},
    {"ticker": "LYV",   "name": "Live Nation Entertainment Inc"},
    {"ticker": "CHTR",  "name": "Charter Communications Inc"},
    {"ticker": "OMC",   "name": "Omnicom Group Inc"},
    {"ticker": "IPG",   "name": "Interpublic Group of Companies Inc"},
    {"ticker": "NYT",   "name": "New York Times Co"},
    {"ticker": "MTCH",  "name": "Match Group Inc"},
    {"ticker": "EDR",   "name": "Endeavor Group Holdings Inc"},
    # Consumer Staples
    {"ticker": "WMT",   "name": "Walmart Inc"},
    {"ticker": "COST",  "name": "Costco Wholesale Corp"},
    {"ticker": "PG",    "name": "Procter & Gamble Co"},
    {"ticker": "KO",    "name": "Coca-Cola Co"},
    {"ticker": "PEP",   "name": "PepsiCo Inc"},
    {"ticker": "PM",    "name": "Philip Morris International Inc"},
    {"ticker": "MO",    "name": "Altria Group Inc"},
    {"ticker": "MDLZ",  "name": "Mondelez International Inc"},
    {"ticker": "CL",    "name": "Colgate-Palmolive Co"},
    {"ticker": "KMB",   "name": "Kimberly-Clark Corp"},
    {"ticker": "KHC",   "name": "Kraft Heinz Co"},
    {"ticker": "GIS",   "name": "General Mills Inc"},
    {"ticker": "K",     "name": "Kellanova"},
    {"ticker": "CPB",   "name": "Campbell Soup Co"},
    {"ticker": "CAG",   "name": "Conagra Brands Inc"},
    {"ticker": "SJM",   "name": "JM Smucker Co"},
    {"ticker": "HRL",   "name": "Hormel Foods Corp"},
    {"ticker": "TSN",   "name": "Tyson Foods Inc"},
    {"ticker": "ADM",   "name": "Archer-Daniels-Midland Co"},
    {"ticker": "BG",    "name": "Bunge Global SA"},
    {"ticker": "STZ",   "name": "Constellation Brands Inc"},
    {"ticker": "BF.B",  "name": "Brown-Forman Corp"},
    {"ticker": "MNST",  "name": "Monster Beverage Corp"},
    {"ticker": "KVUE",  "name": "Kenvue Inc"},
    {"ticker": "CHD",   "name": "Church & Dwight Co Inc"},
    {"ticker": "CLX",   "name": "Clorox Co"},
    {"ticker": "EL",    "name": "Estee Lauder Companies Inc"},
    {"ticker": "SYY",   "name": "Sysco Corp"},
    {"ticker": "USFD",  "name": "US Foods Holding Corp"},
    {"ticker": "PFGC",  "name": "Performance Food Group Co"},
    {"ticker": "LW",    "name": "Lamb Weston Holdings Inc"},
    {"ticker": "TAP",   "name": "Molson Coors Beverage Co"},
    {"ticker": "MKC",   "name": "McCormick & Co Inc"},
    {"ticker": "HBI",   "name": "Hanesbrands Inc"},
    {"ticker": "KR",    "name": "Kroger Co"},
    {"ticker": "WBA",   "name": "Walgreens Boots Alliance Inc"},
    {"ticker": "TGT",   "name": "Target Corp"},
    # Energy
    {"ticker": "XOM",   "name": "Exxon Mobil Corp"},
    {"ticker": "CVX",   "name": "Chevron Corp"},
    {"ticker": "COP",   "name": "ConocoPhillips"},
    {"ticker": "EOG",   "name": "EOG Resources Inc"},
    {"ticker": "SLB",   "name": "Schlumberger NV"},
    {"ticker": "OXY",   "name": "Occidental Petroleum Corp"},
    {"ticker": "PSX",   "name": "Phillips 66"},
    {"ticker": "VLO",   "name": "Valero Energy Corp"},
    {"ticker": "MPC",   "name": "Marathon Petroleum Corp"},
    {"ticker": "HES",   "name": "Hess Corp"},
    {"ticker": "DVN",   "name": "Devon Energy Corp"},
    {"ticker": "FANG",  "name": "Diamondback Energy Inc"},
    {"ticker": "CTRA",  "name": "Coterra Energy Inc"},
    {"ticker": "EQT",   "name": "EQT Corp"},
    {"ticker": "KMI",   "name": "Kinder Morgan Inc"},
    {"ticker": "WMB",   "name": "Williams Companies Inc"},
    {"ticker": "OKE",   "name": "ONEOK Inc"},
    {"ticker": "TRGP",  "name": "Targa Resources Corp"},
    {"ticker": "BKR",   "name": "Baker Hughes Co"},
    {"ticker": "HAL",   "name": "Halliburton Co"},
    {"ticker": "APA",   "name": "APA Corp"},
    {"ticker": "MRO",   "name": "Marathon Oil Corp"},
    # Industrials
    {"ticker": "GE",    "name": "General Electric Co"},
    {"ticker": "CAT",   "name": "Caterpillar Inc"},
    {"ticker": "BA",    "name": "Boeing Co"},
    {"ticker": "UPS",   "name": "United Parcel Service Inc"},
    {"ticker": "RTX",   "name": "RTX Corp"},
    {"ticker": "HON",   "name": "Honeywell International Inc"},
    {"ticker": "UNP",   "name": "Union Pacific Corp"},
    {"ticker": "LMT",   "name": "Lockheed Martin Corp"},
    {"ticker": "DE",    "name": "Deere & Co"},
    {"ticker": "FDX",   "name": "FedEx Corp"},
    {"ticker": "ITW",   "name": "Illinois Tool Works Inc"},
    {"ticker": "MMM",   "name": "3M Co"},
    {"ticker": "EMR",   "name": "Emerson Electric Co"},
    {"ticker": "ETN",   "name": "Eaton Corp"},
    {"ticker": "NOC",   "name": "Northrop Grumman Corp"},
    {"ticker": "GD",    "name": "General Dynamics Corp"},
    {"ticker": "PH",    "name": "Parker-Hannifin Corp"},
    {"ticker": "CMI",   "name": "Cummins Inc"},
    {"ticker": "PCAR",  "name": "PACCAR Inc"},
    {"ticker": "CSX",   "name": "CSX Corp"},
    {"ticker": "NSC",   "name": "Norfolk Southern Corp"},
    {"ticker": "ODFL",  "name": "Old Dominion Freight Line Inc"},
    {"ticker": "JBHT",  "name": "JB Hunt Transport Services Inc"},
    {"ticker": "CHRW",  "name": "CH Robinson Worldwide Inc"},
    {"ticker": "EXPD",  "name": "Expeditors International of Washington Inc"},
    {"ticker": "LUV",   "name": "Southwest Airlines Co"},
    {"ticker": "DAL",   "name": "Delta Air Lines Inc"},
    {"ticker": "UAL",   "name": "United Airlines Holdings Inc"},
    {"ticker": "AAL",   "name": "American Airlines Group Inc"},
    {"ticker": "GWW",   "name": "WW Grainger Inc"},
    {"ticker": "FAST",  "name": "Fastenal Co"},
    {"ticker": "PAYX",  "name": "Paychex Inc"},
    {"ticker": "ADP",   "name": "Automatic Data Processing Inc"},
    {"ticker": "CTAS",  "name": "Cintas Corp"},
    {"ticker": "RSG",   "name": "Republic Services Inc"},
    {"ticker": "WM",    "name": "Waste Management Inc"},
    {"ticker": "XPO",   "name": "XPO Logistics Inc"},
    {"ticker": "LSTR",  "name": "Landstar System Inc"},
    {"ticker": "HII",   "name": "Huntington Ingalls Industries Inc"},
    {"ticker": "TXT",   "name": "Textron Inc"},
    {"ticker": "TDY",   "name": "Teledyne Technologies Inc"},
    {"ticker": "AME",   "name": "Ametek Inc"},
    {"ticker": "IR",    "name": "Ingersoll Rand Inc"},
    {"ticker": "JCI",   "name": "Johnson Controls International PLC"},
    {"ticker": "TT",    "name": "Trane Technologies PLC"},
    {"ticker": "CARR",  "name": "Carrier Global Corp"},
    {"ticker": "OTIS",  "name": "Otis Worldwide Corp"},
    {"ticker": "ALLE",  "name": "Allegion PLC"},
    {"ticker": "AOS",   "name": "Smith (AO) Corp"},
    {"ticker": "LDOS",  "name": "Leidos Holdings Inc"},
    {"ticker": "SAIC",  "name": "Science Applications International Corp"},
    {"ticker": "ROL",   "name": "Rollins Inc"},
    {"ticker": "ROK",   "name": "Rockwell Automation Inc"},
    {"ticker": "DOV",   "name": "Dover Corp"},
    {"ticker": "SWK",   "name": "Stanley Black & Decker Inc"},
    {"ticker": "SNA",   "name": "Snap-on Inc"},
    {"ticker": "TTC",   "name": "Toro Co"},
    {"ticker": "GGG",   "name": "Graco Inc"},
    {"ticker": "NDSN",  "name": "Nordson Corp"},
    {"ticker": "RHI",   "name": "Robert Half Inc"},
    {"ticker": "MAN",   "name": "ManpowerGroup Inc"},
    {"ticker": "J",     "name": "Jacobs Solutions Inc"},
    {"ticker": "EME",   "name": "EMCOR Group Inc"},
    {"ticker": "ACM",   "name": "AECOM"},
    {"ticker": "PNR",   "name": "Pentair PLC"},
    {"ticker": "IEX",   "name": "IDEX Corp"},
    {"ticker": "WAB",   "name": "Westinghouse Air Brake Technologies Corp"},
    {"ticker": "XYL",   "name": "Xylem Inc"},
    {"ticker": "GEV",   "name": "GE Vernova Inc"},
    {"ticker": "GNRC",  "name": "Generac Holdings Inc"},
    {"ticker": "VMC",   "name": "Vulcan Materials Co"},
    {"ticker": "MLM",   "name": "Martin Marietta Materials Inc"},
    {"ticker": "EXP",   "name": "Eagle Materials Inc"},
    {"ticker": "LNX",   "name": "Lennox International Inc"},
    {"ticker": "FIX",   "name": "Comfort Systems USA Inc"},
    {"ticker": "URI",   "name": "United Rentals Inc"},
    {"ticker": "AXON",  "name": "Axon Enterprise Inc"},
    {"ticker": "HUBB",  "name": "Hubbell Inc"},
    # Materials
    {"ticker": "LIN",   "name": "Linde PLC"},
    {"ticker": "APD",   "name": "Air Products and Chemicals Inc"},
    {"ticker": "ECL",   "name": "Ecolab Inc"},
    {"ticker": "SHW",   "name": "Sherwin-Williams Co"},
    {"ticker": "PPG",   "name": "PPG Industries Inc"},
    {"ticker": "NEM",   "name": "Newmont Corp"},
    {"ticker": "FCX",   "name": "Freeport-McMoRan Inc"},
    {"ticker": "NUE",   "name": "Nucor Corp"},
    {"ticker": "STLD",  "name": "Steel Dynamics Inc"},
    {"ticker": "RS",    "name": "Reliance Steel & Aluminum Co"},
    {"ticker": "DOW",   "name": "Dow Inc"},
    {"ticker": "DD",    "name": "DuPont de Nemours Inc"},
    {"ticker": "LYB",   "name": "LyondellBasell Industries NV"},
    {"ticker": "WLK",   "name": "Westlake Corp"},
    {"ticker": "CE",    "name": "Celanese Corp"},
    {"ticker": "EMN",   "name": "Eastman Chemical Co"},
    {"ticker": "ALB",   "name": "Albemarle Corp"},
    {"ticker": "IFF",   "name": "International Flavors & Fragrances Inc"},
    {"ticker": "CTVA",  "name": "Corteva Inc"},
    {"ticker": "FMC",   "name": "FMC Corp"},
    {"ticker": "MOS",   "name": "Mosaic Co"},
    {"ticker": "CF",    "name": "CF Industries Holdings Inc"},
    {"ticker": "PKG",   "name": "Packaging Corp of America"},
    {"ticker": "WRK",   "name": "WestRock Co"},
    {"ticker": "IP",    "name": "International Paper Co"},
    # Real Estate
    {"ticker": "PLD",   "name": "Prologis Inc"},
    {"ticker": "EQIX",  "name": "Equinix Inc"},
    {"ticker": "AMT",   "name": "American Tower Corp"},
    {"ticker": "DLR",   "name": "Digital Realty Trust Inc"},
    {"ticker": "CCI",   "name": "Crown Castle International Corp"},
    {"ticker": "PSA",   "name": "Public Storage"},
    {"ticker": "WELL",  "name": "Welltower Inc"},
    {"ticker": "AVB",   "name": "AvalonBay Communities Inc"},
    {"ticker": "EQR",   "name": "Equity Residential"},
    {"ticker": "ESS",   "name": "Essex Property Trust Inc"},
    {"ticker": "MAA",   "name": "Mid-America Apartment Communities Inc"},
    {"ticker": "INVH",  "name": "Invitation Homes Inc"},
    {"ticker": "SUI",   "name": "Sun Communities Inc"},
    {"ticker": "UDR",   "name": "UDR Inc"},
    {"ticker": "VTR",   "name": "Ventas Inc"},
    {"ticker": "ARE",   "name": "Alexandria Real Estate Equities Inc"},
    {"ticker": "BXP",   "name": "Boston Properties Inc"},
    {"ticker": "SPG",   "name": "Simon Property Group Inc"},
    {"ticker": "KIM",   "name": "Kimco Realty Corp"},
    {"ticker": "REG",   "name": "Regency Centers Corp"},
    {"ticker": "FRT",   "name": "Federal Realty Investment Trust"},
    {"ticker": "O",     "name": "Realty Income Corp"},
    {"ticker": "IRM",   "name": "Iron Mountain Inc"},
    {"ticker": "WY",    "name": "Weyerhaeuser Co"},
    {"ticker": "HST",   "name": "Host Hotels & Resorts Inc"},
    {"ticker": "PEAK",  "name": "Healthpeak Properties Inc"},
    {"ticker": "DOC",   "name": "Physicians Realty Trust"},
    {"ticker": "CUZ",   "name": "Cousins Properties Inc"},
    {"ticker": "KRC",   "name": "Kilroy Realty Corp"},
    {"ticker": "SLG",   "name": "SL Green Realty Corp"},
    {"ticker": "VICI",  "name": "VICI Properties Inc"},
    # Utilities
    {"ticker": "NEE",   "name": "NextEra Energy Inc"},
    {"ticker": "DUK",   "name": "Duke Energy Corp"},
    {"ticker": "SO",    "name": "Southern Co"},
    {"ticker": "AEP",   "name": "American Electric Power Co Inc"},
    {"ticker": "EXC",   "name": "Exelon Corp"},
    {"ticker": "SRE",   "name": "Sempra"},
    {"ticker": "PEG",   "name": "Public Service Enterprise Group Inc"},
    {"ticker": "XEL",   "name": "Xcel Energy Inc"},
    {"ticker": "ED",    "name": "Consolidated Edison Inc"},
    {"ticker": "EIX",   "name": "Edison International"},
    {"ticker": "WEC",   "name": "WEC Energy Group Inc"},
    {"ticker": "PPL",   "name": "PPL Corp"},
    {"ticker": "DTE",   "name": "DTE Energy Co"},
    {"ticker": "ETR",   "name": "Entergy Corp"},
    {"ticker": "FE",    "name": "FirstEnergy Corp"},
    {"ticker": "AEE",   "name": "Ameren Corp"},
    {"ticker": "CMS",   "name": "CMS Energy Corp"},
    {"ticker": "CNP",   "name": "CenterPoint Energy Inc"},
    {"ticker": "LNT",   "name": "Alliant Energy Corp"},
    {"ticker": "NI",    "name": "NiSource Inc"},
    {"ticker": "EVRG",  "name": "Evergy Inc"},
    {"ticker": "OGE",   "name": "OGE Energy Corp"},
    {"ticker": "ATO",   "name": "Atmos Energy Corp"},
    {"ticker": "AES",   "name": "AES Corp"},
    {"ticker": "NRG",   "name": "NRG Energy Inc"},
    {"ticker": "VST",   "name": "Vistra Corp"},
    {"ticker": "CEG",   "name": "Constellation Energy Corp"},
    {"ticker": "UGI",   "name": "UGI Corp"},
    {"ticker": "NFG",   "name": "National Fuel Gas Co"},
    {"ticker": "PNW",   "name": "Pinnacle West Capital Corp"},
    {"ticker": "AWK",   "name": "American Water Works Co Inc"},



    ##########################


    {"ticker": "ACN",  "name": "Accenture PLC"},             # was always S&P 500
    {"ticker": "APP",  "name": "AppLovin Corp"},             # added 2025
    {"ticker": "COHR", "name": "Coherent Corp"},             # added Mar 2026
    {"ticker": "CTSH", "name": "Cognizant Technology Solutions Corp"},
    {"ticker": "FICO", "name": "Fair Isaac Corp"},
    {"ticker": "GDDY", "name": "GoDaddy Inc"},
    {"ticker": "LITE", "name": "Lumentum Holdings Inc"},
    {"ticker": "XYZ",  "name": "Block Inc"},                 # formerly SQ; rebranded

    # ── Financials (12) ──────────────────────────────────────────────────
    {"ticker": "ACGL", "name": "Arch Capital Group Ltd"},
    {"ticker": "AON",  "name": "Aon PLC"},
    {"ticker": "ARES", "name": "Ares Management Corp"},      # added late 2024
    {"ticker": "COIN", "name": "Coinbase Global Inc"},       # added May 2025
    {"ticker": "FI",   "name": "Fiserv Inc"},                # note: original list has FISV (old ticker)
    {"ticker": "FLT",  "name": "Corpay Inc"},                # formerly FleetCor Technologies
    {"ticker": "HIG",  "name": "Hartford Financial Services Group Inc"},
    {"ticker": "HOOD", "name": "Robinhood Markets Inc"},     # added 2025
    {"ticker": "IBKR", "name": "Interactive Brokers Group Inc"},
    {"ticker": "PYPL", "name": "PayPal Holdings Inc"},
    {"ticker": "ACGL", "name": "Arch Capital Group Ltd"},    # (already listed above — dedup if needed)

    # ── Industrials (7) ──────────────────────────────────────────────────
    {"ticker": "CPRT", "name": "Copart Inc"},
    {"ticker": "EFX",  "name": "Equifax Inc"},
    {"ticker": "FTV",  "name": "Fortive Corp"},
    {"ticker": "LHX",  "name": "L3Harris Technologies Inc"},
    {"ticker": "PWR",  "name": "Quanta Services Inc"},
    {"ticker": "TDG",  "name": "TransDigm Group Inc"},
    {"ticker": "VRSK", "name": "Verisk Analytics Inc"},

    # ── Consumer Discretionary (2) ────────────────────────────────────────
    {"ticker": "DASH", "name": "DoorDash Inc"},              # added 2025
    {"ticker": "WSM",  "name": "Williams-Sonoma Inc"},       # added 2025

    # ── Consumer Staples (3) ─────────────────────────────────────────────
    {"ticker": "CASY", "name": "Casey's General Stores Inc"},# added Apr 2026
    {"ticker": "HSY",  "name": "Hershey Co"},
    {"ticker": "KDP",  "name": "Keurig Dr Pepper Inc"},

    # ── Communication Services (2) ────────────────────────────────────────
    {"ticker": "SATS", "name": "EchoStar Corp"},             # added Mar 2026
    {"ticker": "TKO",  "name": "TKO Group Holdings Inc"},    # added 2025

    # ── Energy (2) ───────────────────────────────────────────────────────
    {"ticker": "EXE",  "name": "Expand Energy Corp"},        # added 2025 (replaced Chesapeake)
    {"ticker": "TPL",  "name": "Texas Pacific Land Corp"},

    # ── Real Estate (5) ──────────────────────────────────────────────────
    {"ticker": "CBRE", "name": "CBRE Group Inc"},
    {"ticker": "CSGP", "name": "CoStar Group Inc"},
    {"ticker": "EXR",  "name": "Extra Space Storage Inc"},
    {"ticker": "SBAC", "name": "SBA Communications Corp"},

    # ── Utilities (3) ────────────────────────────────────────────────────
    {"ticker": "D",    "name": "Dominion Energy Inc"},
    {"ticker": "ES",   "name": "Eversource Energy"},
    {"ticker": "PCG",  "name": "PG&E Corp"},

    # ── Health Care (1) ──────────────────────────────────────────────────
    {"ticker": "ZBH",  "name": "Zimmer Biomet Holdings Inc"},

    # ── Materials (1) ─────────────────────────────────────────────────────
    {"ticker": "BALL", "name": "Ball Corp"},

    # extra
    {"ticker": "AMCR", "name": "Amcor PLC"},
    {"ticker": "APO",  "name": "Apollo Global Management Inc"},
    {"ticker": "AVY",  "name": "Avery Dennison Corp"},
    {"ticker": "CIEN", "name": "Ciena Corp"},
    {"ticker": "CPAY", "name": "Corpay Inc"},
    {"ticker": "CPT",  "name": "Camden Property Trust"},
    {"ticker": "CRH",  "name": "CRH PLC"},
    {"ticker": "EG",   "name": "Everest Group Ltd"},
    {"ticker": "EXPE", "name": "Expedia Group Inc"},
    {"ticker": "KKR",  "name": "KKR & Co Inc"},
    {"ticker": "LII",  "name": "Lennox International Inc"},
    {"ticker": "MAS",  "name": "Masco Corp"},
    {"ticker": "MRSH", "name": "Marsh & McLennan Companies Inc"},
    {"ticker": "NCLH", "name": "Norwegian Cruise Line Holdings Ltd"},
    {"ticker": "PSKY", "name": "Paramount Skydance Corp"},
    {"ticker": "Q",    "name": "Qurate Retail Inc"},
    {"ticker": "SNDK", "name": "Sandisk Corp"},
    {"ticker": "SW",   "name": "Smurfit WestRock PLC"},
    {"ticker": "TRMB", "name": "Trimble Inc"},
    {"ticker": "UBER", "name": "Uber Technologies Inc"},
    {"ticker": "VEEV", "name": "Veeva Systems Inc"},
    {"ticker": "VLTO", "name": "Veralto Corp"},
    {"ticker": "VRT",  "name": "Vertiv Holdings Co"},

]

# Hardcoded CIK map
KNOWN_CIKS = {
    "BRK.B": "0001067983", "BRK.A": "0001067983",
    "BF.B":  "0000014693", "BF.A":  "0000014693",
    "AAPL":  "0000320193", "MSFT":  "0000789019", "NVDA":  "0001045810",
    "AVGO":  "0001730168", "ORCL":  "0001341439", "ADBE":  "0000796343",
    "CRM":   "0001108524", "AMD":   "0000002488", "INTC":  "0000050863",
    "QCOM":  "0000804328", "TXN":   "0000097476", "IBM":   "0000051143",
    "CSCO":  "0000858877", "NOW":   "0001373715", "PANW":  "0001327567",
    "INTU":  "0000896878", "SNOW":  "0001640147", "CRWD":  "0001535527",
    "DELL":  "0001571123", "HPQ":   "0000047217", "GOOGL": "0001652044",
    "GOOG":  "0001652044", "META":  "0001326801", "AMZN":  "0001018724",
    "TSLA":  "0001318605", "NFLX":  "0001065280", "JPM":   "0000019617",
    "BAC":   "0000070858", "WFC":   "0000072971", "GS":    "0000886982",
    "MS":    "0000895421", "V":     "0001403161", "MA":    "0001141391",
    "WMT":   "0000104169", "COST":  "0000909832", "HD":    "0000354950",
    "LLY":   "0000059478", "JNJ":   "0000200406", "UNH":   "0000731766",
    "MRK":   "0000310158", "ABBV":  "0001551152", "PFE":   "0000078003",
    "XOM":   "0000034088", "CVX":   "0000093410", "GE":    "0000040533",
    "CAT":   "0000018230", "BA":    "0000012927", "UPS":   "0001090727",
    "DIS":   "0001001039", "T":     "0000732717", "VZ":    "0000732712",
    "KO":    "0000021344", "PEP":   "0000077476", "PG":    "0000080424",
    "NEE":   "0000753308", "DUK":   "0001326428", "SO":    "0000092122",

    "BRK.B":  "0001067983", "BRK.A":  "0001067983",
    "BRK-B":  "0001067983", "BRK-A":  "0001067983",
    "BF.B":   "0000014693", "BF.A":   "0000014693",
    "BF-B":   "0000014693", "BF-A":   "0000014693",
    # Technology
    "AAPL":   "0000320193", "MSFT":   "0000789019", "NVDA":   "0001045810",
    "AVGO":   "0001730168", "ORCL":   "0001341439", "ADBE":   "0000796343",
    "CRM":    "0001108524", "AMD":    "0000002488", "INTC":   "0000050863",
    "QCOM":   "0000804328", "TXN":    "0000097476", "IBM":    "0000051143",
    "CSCO":   "0000858877", "NOW":    "0001373715", "PANW":   "0001327567",
    "INTU":   "0000896878", "SNOW":   "0001640147", "CRWD":   "0001535527",
    "DELL":   "0001571123", "HPQ":    "0000047217", "ADI":    "0000006281",
    "MU":     "0000723125", "LRCX":   "0000707549", "KLAC":   "0000319201",
    "AMAT":   "0000006951", "MCHP":   "0000827054", "ON":     "0000863435",
    "FTNT":   "0001262039", "NET":    "0001660134", "ZS":     "0001713683",
    "DDOG":   "0001561180", "MDB":    "0001441816", "TEAM":   "0001390502",
    "WDAY":   "0001327811", "ADSK":   "0000769397", "CDNS":   "0000813672",
    "SNPS":   "0000883241", "ROP":    "0000882184", "TTD":    "0001671933",
    "HUBS":   "0001404655", "AKAM":   "0001086222", "VRSN":   "0001014473",
    "FFIV":   "0001048695", "JNPR":   "0001043604", "MSI":    "0000068505",
    "HPE":    "0001645590", "CDW":    "0001418302", "GLW":    "0000024741",
    "APH":    "0000820313", "TEL":    "0001385187", "JBL":    "0000898293",
    "FLEX":   "0000866374", "STX":    "0001137789", "WDC":    "0000106040",
    "NTAP":   "0001002047", "KEYS":   "0001601712", "MPWR":   "0001280452",
    "SWKS":   "0000004127", "NXPI":   "0001413447", "ANET":   "0001313925",
    "IT":     "0000749251", "GEN":    "0000849399", "EPAM":   "0001352010",
    "TYL":    "0000860731", "PTC":    "0000857005", "ANSS":   "0000820081",
    "SSNC":   "0000883948", "ZM":     "0001585521", "DOCU":   "0001261333",
    "OKTA":   "0001660134", "SMCI":   "0001375365", "PLTR":   "0001321655",
    "FSLR":   "0001274494", "ENPH":   "0001463101", "TER":    "0000097210",
    "VSH":    "0000103730", "GL":     "0000098752", "ZBRA":   "0000877212",
    # Financials
    "JPM":    "0000019617", "V":      "0001403161", "MA":     "0001141391",
    "BAC":    "0000070858", "WFC":    "0000072971", "GS":     "0000886982",
    "MS":     "0000895421", "C":      "0000831001", "AXP":    "0000004846",
    "BLK":    "0001364742", "SCHW":   "0000316709", "SPGI":   "0000064040",
    "ICE":    "0001571049", "MCO":    "0001059556", "CME":    "0001156375",
    "PNC":    "0000713676", "USB":    "0000036104", "TFC":    "0000092122",
    "COF":    "0000927628", "BK":     "0000009626", "STT":    "0000093751",
    "FITB":   "0000035527", "KEY":    "0000091576", "CFG":    "0001601712",
    "HBAN":   "0000049196", "RF":     "0001281761", "SYF":    "0001601712",
    "DFS":    "0001393612", "MTB":    "0000036270", "FIS":    "0000798354",
    "FISV":   "0000798354", "GPN":    "0001123360", "PGR":    "0000080661",
    "ALL":    "0000899051", "TRV":    "0000086312", "AIG":    "0000005272",
    "MET":    "0001099219", "PRU":    "0001137774", "CB":     "0000896159",
    "AFL":    "0000004977", "AJG":    "0000354190", "MMC":    "0000062234",
    "WTW":    "0001140536", "BRO":    "0000009984", "CINF":   "0000020286",
    "WRB":    "0000011544", "PFG":    "0001126328", "TROW":   "0001113169",
    "BEN":    "0000038777", "AMP":    "0000820081", "RJF":    "0000720672",
    "NTRS":   "0000073124", "L":      "0000060714", "UNM":    "0000005513",
    "AIZ":    "0001267238", "FDS":    "0000040570", "MKTX":   "0001278021",
    "NDAQ":   "0001120193", "CBOE":   "0001374310", "LPLA":   "0001397911",
    "IVZ":    "0000914208", "CMA":    "0000028412", "ZION":   "0000109380",
    "ERIE":   "0000049697", "JKHY":   "0000896429", "BR":     "0001383312",
    "MSCI":   "0001408198", "BLDR":   "0001571049", "BX":     "0001393818",
    # Healthcare
    "LLY":    "0000059478", "JNJ":    "0000200406", "UNH":    "0000731766",
    "MRK":    "0000310158", "ABBV":   "0001551152", "PFE":    "0000078003",
    "TMO":    "0000097476", "ABT":    "0000001800", "DHR":    "0000313616",
    "REGN":   "0000872589", "VRTX":   "0000875320", "ISRG":   "0001035267",
    "BSX":    "0000885978", "SYK":    "0000310764", "MDT":    "0001613103",
    "CI":     "0001739940", "BMY":    "0000014272", "GILD":   "0000882095",
    "AMGN":   "0000820081", "CVS":    "0000064803", "ELV":    "0001156039",
    "HUM":    "0000049071", "MRNA":   "0001682852", "BIIB":   "0000875045",
    "ZTS":    "0001555280", "IQV":    "0001478930", "LH":     "0000920148",
    "DGX":    "0000920148", "CAH":    "0000721371", "MCK":    "0000927653",
    "COR":    "0001140859", "BAX":    "0000010456", "BDX":    "0000010795",
    "GEHC":   "0001879273", "RMD":    "0000943819", "STE":    "0001060349",
    "HCA":    "0000860730", "CNC":    "0001071739", "HOLX":   "0000859737",
    "IDXX":   "0000874716", "MTD":    "0001037785", "WST":    "0000105770",
    "EW":     "0001099800", "DXCM":   "0001274515", "ALGN":   "0001097149",
    "INCY":   "0000879169", "PODD":   "0001145986", "MASI":   "0000937556",
    "CRL":    "0001100682", "TECH":   "0001109189", "MEDP":   "0001611647",
    "BRKR":   "0001109189", "WAT":    "0001000697", "A":      "0001090872",
    "RVTY":   "0000031791", "COO":    "0000723254", "HWM":    "0000004281",
    "UHS":    "0000352915", "VTRS":   "0001792789", "MOH":    "0001179929",
    "DVA":    "0000927066", "XRAY":   "0000818479", "HSIC":   "0001000228",
    "SOLV":   "0001939200", "BNTX":   "0001776197",
    # Consumer Discretionary
    "AMZN":   "0001018724", "TSLA":   "0001318605", "HD":     "0000354950",
    "LOW":    "0000060667", "SBUX":   "0000829224", "MCD":    "0000063908",
    "BKNG":   "0001075531", "TJX":    "0000109198", "NKE":    "0000320187",
    "ABNB":   "0001559720", "CMG":    "0001058090", "MAR":    "0001048268",
    "HLT":    "0001585583", "RCL":    "0000884887", "CCL":    "0000723254",
    "DHI":    "0000045012", "LEN":    "0000920760", "NVR":    "0000906163",
    "PHM":    "0000822818", "TOL":    "0000794170", "F":      "0000037996",
    "GM":     "0001467858", "APTV":   "0001521332", "BWA":    "0000908255",
    "LKQ":    "0001065696", "GPC":    "0000040987", "DRI":    "0000940944",
    "YUM":    "0001041514", "DPZ":    "0001286681", "QSR":    "0001618673",
    "TSCO":   "0000916365", "ULTA":   "0001403568", "DG":     "0000029534",
    "DLTR":   "0000935703", "ROST":   "0000745732", "EBAY":   "0001065088",
    "AZO":    "0000866787", "ORLY":   "0000898173", "BBY":    "0000764180",
    "POOL":   "0000945841", "HAS":    "0000046080", "MAT":    "0000063276",
    "ETSY":   "0001370637", "CVNA":   "0001690820", "WYNN":   "0001174922",
    "LVS":    "0001300514", "MGM":    "0000789570", "CZR":    "0000858339",
    "TPR":    "0001116132", "PVH":    "0000078239", "RL":     "0001037038",
    "DECK":   "0000895655", "GRMN":   "0001121788", "LULU":   "0001397187",
    # Communication Services
    "META":   "0001326801", "GOOGL":  "0001652044", "GOOG":   "0001652044",
    "NFLX":   "0001065280", "DIS":    "0001001039", "CMCSA":  "0001166691",
    "T":      "0000732717", "VZ":     "0000732712", "TMUS":   "0001283699",
    "EA":     "0000712034", "TTWO":   "0000946581", "WBD":    "0001437107",
    "PARA":   "0000813828", "FOXA":   "0001308161", "FOX":    "0001308161",
    "NWSA":   "0001564708", "NWS":    "0001564708", "LYV":    "0001335258",
    "CHTR":   "0001091334", "OMC":    "0000029989", "IPG":    "0000397187",
    "NYT":    "0000071691", "MTCH":   "0000891482", "EDR":    "0001817040",
    # Consumer Staples
    "WMT":    "0000104169", "COST":   "0000909832", "PG":     "0000080424",
    "KO":     "0000021344", "PEP":    "0000077476", "PM":     "0001413159",
    "MO":     "0000101830", "MDLZ":   "0001418135", "CL":     "0000021665",
    "KMB":    "0000055785", "KHC":    "0001637459", "GIS":    "0000040704",
    "K":      "0000055529", "CPB":    "0000016732", "CAG":    "0000023217",
    "SJM":    "0000091419", "HRL":    "0000048465", "TSN":    "0000100493",
    "ADM":    "0000007084", "BG":     "0001144519", "STZ":    "0000016918",
    "MNST":   "0000865752", "KVUE":   "0001945579", "CHD":    "0000313927",
    "CLX":    "0000021076", "EL":     "0001001250", "SYY":    "0000086312",
    "USFD":   "0001665002", "PFGC":   "0001174954", "LW":     "0001679273",
    "TAP":    "0000024545", "MKC":    "0000063754", "HBI":    "0001359841",
    "KR":     "0000056873", "WBA":    "0001511288", "TGT":    "0000027419",
    # Energy
    "XOM":    "0000034088", "CVX":    "0000093410", "COP":    "0001163165",
    "EOG":    "0000821189", "SLB":    "0000087347", "OXY":    "0000797468",
    "PSX":    "0001534701", "VLO":    "0001035002", "MPC":    "0001510295",
    "HES":    "0000004447", "DVN":    "0001090012", "FANG":   "0001539838",
    "CTRA":   "0001071739", "EQT":    "0000033213", "KMI":    "0001110805",
    "WMB":    "0000107263", "OKE":    "0001040792", "TRGP":   "0001389170",
    "BKR":    "0001701605", "HAL":    "0000045012", "APA":    "0000006769",
    "MRO":    "0000101778",
    # Industrials
    "GE":     "0000040533", "CAT":    "0000018230", "BA":     "0000012927",
    "UPS":    "0001090727", "RTX":    "0000101829", "HON":    "0000773840",
    "UNP":    "0000100885", "LMT":    "0000936468", "DE":     "0000315189",
    "FDX":    "0001048911", "ITW":    "0000049826", "MMM":    "0000066740",
    "EMR":    "0000032604", "ETN":    "0001551182", "NOC":    "0001133421",
    "GD":     "0000040533", "PH":     "0000076334", "CMI":    "0000026172",
    "PCAR":   "0000075362", "CSX":    "0000277948", "NSC":    "0000702165",
    "ODFL":   "0000878927", "JBHT":   "0000728535", "CHRW":   "0001043277",
    "EXPD":   "0000765880", "LUV":    "0000092380", "DAL":    "0000027904",
    "UAL":    "0000100517", "AAL":    "0000004515", "GWW":    "0000277135",
    "FAST":   "0000815556", "PAYX":   "0000723531", "ADP":    "0000012063",
    "CTAS":   "0000723254", "RSG":    "0001060349", "WM":     "0000823768",
    "XPO":    "0001610670", "LSTR":   "0000853816", "HII":    "0001501585",
    "TXT":    "0000217346", "TDY":    "0001039399", "AME":    "0000014846",
    "IR":     "0001466258", "JCI":    "0001477294", "TT":     "0001466258",
    "CARR":   "0001783180", "OTIS":   "0001781335", "ALLE":   "0001579241",
    "AOS":    "0000091142", "LDOS":   "0001336920", "SAIC":   "0001336920",
    "ROL":    "0000085408", "ROK":    "0001024478", "DOV":    "0000029905",
    "SWK":    "0000093556", "SNA":    "0000091440", "TTC":    "0000098362",
    "GGG":    "0000850693", "NDSN":   "0000073366", "RHI":    "0000315213",
    "MAN":    "0000871763", "J":      "0001466258", "EME":    "0000790816",
    "ACM":    "0000868780", "PNR":    "0000077360", "IEX":    "0000832101",
    "WAB":    "0000943452", "XYL":    "0001524472", "GEV":    "0001879407",
    "GNRC":   "0001474735", "VMC":    "0001396009", "MLM":    "0000916076",
    "EXP":    "0000918646", "LNX":    "0000059527", "FIX":    "0000784977",
    "URI":    "0001067294", "AXON":   "0001069183", "HUBB":   "0000048898",
    # Materials
    "LIN":    "0001707092", "APD":    "0000002969", "ECL":    "0000031791",
    "SHW":    "0000089089", "PPG":    "0000079879", "NEM":    "0001164180",
    "FCX":    "0000831259", "NUE":    "0000073309", "STLD":   "0001022671",
    "RS":     "0000930420", "DOW":    "0001751788", "DD":     "0001666700",
    "LYB":    "0001489393", "WLK":    "0001286043", "CE":     "0000813672",
    "EMN":    "0000915389", "ALB":    "0000915779", "IFF":    "0000049600",
    "CTVA":   "0001755672", "FMC":    "0000037785", "MOS":    "0001285785",
    "CF":     "0001270391", "PKG":    "0000075829", "WRK":    "0001732845",
    "IP":     "0000049826",
    # Real Estate
    "PLD":    "0001045609", "EQIX":   "0001101239", "AMT":    "0001053507",
    "DLR":    "0001297996", "CCI":    "0001051512", "PSA":    "0001393311",
    "WELL":   "0000766704", "AVB":    "0000915912", "EQR":    "0000906107",
    "ESS":    "0000920522", "MAA":    "0000912593", "INVH":   "0001687229",
    "SUI":    "0000912595", "UDR":    "0000074260", "VTR":    "0000740260",
    "ARE":    "0001043186", "BXP":    "0001037540", "SPG":    "0001063761",
    "KIM":    "0000879585", "REG":    "0000910606", "FRT":    "0000034903",
    "O":      "0000726854", "IRM":    "0001020569", "WY":     "0000106535",
    "HST":    "0001070750", "PEAK":   "0000765880", "DOC":    "0001612630",
    "CUZ":    "0000025232", "KRC":    "0001040971", "SLG":    "0001040971",
    "VICI":   "0001695678",
    # Utilities
    "NEE":    "0000753308", "DUK":    "0001326428", "SO":     "0000092122",
    "AEP":    "0000004904", "EXC":    "0000008192", "SRE":    "0001032975",
    "PEG":    "0000081033", "XEL":    "0000072741", "ED":     "0001047862",
    "EIX":    "0000827187", "WEC":    "0000783325", "PPL":    "0000922224",
    "DTE":    "0000936340", "ETR":    "0000049600", "FE":     "0001031296",
    "AEE":    "0001002910", "CMS":    "0000811156", "CNP":    "0001130310",
    "LNT":    "0000052485", "NI":     "0001111711", "EVRG":   "0001711269",
    "OGE":    "0000074260", "ATO":    "0000731802", "AES":    "0000874761",
    "NRG":    "0001013871", "VST":    "0001692819", "CEG":    "0001868275",
    "UGI":    "0000101929", "NFG":    "0000070858", "PNW":    "0000007286",
    "AWK":    "0001410636",


    ##############################

    # ── Information Technology ────────────────────────────────────────────────
    "ACN":   "0001467373",  # Accenture PLC
    "APP":   "0001751008",  # AppLovin Corp
    "COHR":  "0000820318",  # Coherent Corp (formerly II-VI Inc)
    "CTSH":  "0001058290",  # Cognizant Technology Solutions Corp
    "FICO":  "0000814547",  # Fair Isaac Corp
    "GDDY":  "0001609711",  # GoDaddy Inc
    "LITE":  "0001616862",  # Lumentum Holdings Inc
    "XYZ":   "0001512673",  # Block Inc (formerly Square Inc)

    # ── Financials ────────────────────────────────────────────────────────────
    "ACGL":  "0000947484",  # Arch Capital Group Ltd
    "AON":   "0000315293",  # Aon PLC
    "ARES":  "0001555280",  # Ares Management Corp
    "COIN":  "0001679788",  # Coinbase Global Inc
    "FI":    "0000798354",  # Fiserv Inc (ticker changed from FISV)
    "FLT":   "0001175922",  # Corpay Inc (formerly FleetCor Technologies)
    "HIG":   "0000874766",  # Hartford Financial Services Group Inc
    "HOOD":  "0001783398",  # Robinhood Markets Inc
    "IBKR":  "0001381197",  # Interactive Brokers Group Inc
    "PYPL":  "0001633917",  # PayPal Holdings Inc

    # ── Industrials ───────────────────────────────────────────────────────────
    "CPRT":  "0000900075",  # Copart Inc
    "EFX":   "0000033185",  # Equifax Inc
    "FTV":   "0001659166",  # Fortive Corp
    "LHX":   "0000202058",  # L3Harris Technologies Inc
    "PWR":   "0001050915",  # Quanta Services Inc
    "TDG":   "0001260221",  # TransDigm Group Inc
    "VRSK":  "0001442145",  # Verisk Analytics Inc

    # ── Consumer Discretionary ────────────────────────────────────────────────
    "DASH":  "0001792789",  # DoorDash Inc
    "WSM":   "0001077712",  # Williams-Sonoma Inc

    # ── Consumer Staples ──────────────────────────────────────────────────────
    "CASY":  "0000726958",  # Casey's General Stores Inc
    "HSY":   "0000047111",  # Hershey Co
    "KDP":   "0001418135",  # Keurig Dr Pepper Inc

    # ── Communication Services ────────────────────────────────────────────────
    "SATS":  "0001415404",  # EchoStar Corp
    "TKO":   "0001885538",  # TKO Group Holdings Inc

    # ── Energy ────────────────────────────────────────────────────────────────
    "EXE":   "0001070154",  # Expand Energy Corp (formerly Chesapeake Energy)
    "TPL":   "0001811074",  # Texas Pacific Land Corp

    # ── Real Estate ───────────────────────────────────────────────────────────
    "CBRE":  "0001138118",  # CBRE Group Inc
    "CSGP":  "0001057352",  # CoStar Group Inc
    "EXR":   "0001289490",  # Extra Space Storage Inc
    "SBAC":  "0001034669",  # SBA Communications Corp

    # ── Utilities ─────────────────────────────────────────────────────────────
    "D":     "0000715072",  # Dominion Energy Inc
    "ES":    "0000072741",  # Eversource Energy
    "PCG":   "0001004440",  # PG&E Corp

    # ── Health Care ───────────────────────────────────────────────────────────
    "ZBH":   "0001136893",  # Zimmer Biomet Holdings Inc

    # ── Materials ─────────────────────────────────────────────────────────────
    "BALL":  "0000009389",  # Ball Corp

    # extra
    "AMCR": "0001748790", "APO": "0001858681", "AVY": "0000008818",
    "CIEN": "0000936395", "CPAY": "0001175454", "CPT": "0000906345",
    "CRH": "0001787017", "EG": "0001095073", "EXPE": "0001324424",
    "KKR": "0001404912", "LII": "0001069202", "MAS": "0000062996",
    "MRSH": "0000062709", "NCLH": "0001513761", "PSKY": "CIK_NOT_FOUND",
    "Q": "0001564708", "SNDK": "0001000184", "SW": "0002005951",
    "TRMB": "0000864749", "UBER": "0001543151", "VEEV": "0001393052",
    "VLTO": "0001967680", "VRT": "0001697776",

}


# ─── EDGAR SESSION ────────────────────────────────────────────────────────────

class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE  = "https://www.sec.gov"

    def __init__(self):
        self._session = self._build()

    def _build(self) -> requests.Session:
        sess  = requests.Session()
        retry = Retry(
            total            = MAX_RETRIES,
            backoff_factor   = BACKOFF_BASE,
            status_forcelist = [429, 500, 502, 503, 504],
            allowed_methods  = ["GET", "HEAD"],
            raise_on_status  = False,
        )
        sess.mount("https://", HTTPAdapter(max_retries=retry))
        sess.mount("http://",  HTTPAdapter(max_retries=retry))
        return sess

    def get(self, url: str, timeout: int = 30) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        headers = {
            "User-Agent":      _next_ua(),
            "Accept-Encoding": "gzip, deflate, br",
            "Accept":          "application/json, text/html, */*",
        }
        for attempt in range(MAX_RETRIES):
            try:
                r = self._session.get(url, headers=headers, timeout=timeout)
                if r.status_code == 429:
                    wait = min(BACKOFF_BASE ** (attempt + 2) * (0.8 + random.random() * 0.4), BACKOFF_MAX)
                    print(f"    [429] Rate limited — waiting {wait:.0f}s …")
                    time.sleep(wait)
                    continue
                if r.status_code in (403, 404):
                    return None
                r.raise_for_status()
                return r
            except requests.exceptions.Timeout:
                time.sleep(min(BACKOFF_BASE ** (attempt + 1), BACKOFF_MAX))
            except Exception as e:
                print(f"    [ERR] {e} | {url[-70:]}")
                return None
        return None

    def reset(self):
        try:    self._session.close()
        except: pass
        self._session = self._build()


# ─── CIK LOOKUP ──────────────────────────────────────────────────────────────

_tickers_cache: Optional[dict] = None
_cik_cache:     dict           = {}
_cik_lock                      = threading.Lock()


def resolve_cik(ticker: str, sess: EDGARSession) -> Optional[str]:
    global _tickers_cache

    key = ticker.upper()
    if key in KNOWN_CIKS:
        return KNOWN_CIKS[key]

    with _cik_lock:
        if ticker in _cik_cache:
            return _cik_cache[ticker]
        if _tickers_cache is None:
            r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
            _tickers_cache = r.json() if r else {}

    variants = [key]
    if "." in key:
        b, c = key.rsplit(".", 1)
        variants += [f"{b}-{c}", f"{b}{c}"]
    elif "-" in key:
        b, c = key.rsplit("-", 1)
        variants += [f"{b}.{c}", f"{b}{c}"]

    for variant in variants:
        for entry in (_tickers_cache or {}).values():
            if entry.get("ticker", "").upper() == variant:
                cik = str(entry["cik_str"]).zfill(10)
                with _cik_lock:
                    _cik_cache[ticker] = cik
                return cik

    print(f"  [WARN] No CIK found for {ticker}")
    return None


# ─── FILING DISCOVERY ────────────────────────────────────────────────────────

def get_target_filings(cik: str, sess: EDGARSession, years_back: int = YEARS_BACK) -> list:
    """
    Returns list of target filings within the past `years_back` years.
    Each entry: { form_type, filed_date, period_of_report, accession_no, primary_document }
    """
    url = f"{sess.DATA_BASE}/submissions/CIK{cik}.json"
    r   = sess.get(url)
    if not r:
        return []

    data   = r.json()
    cutoff = (datetime.now() - timedelta(days=years_back * 365)).strftime("%Y-%m-%d")
    rows   = []

    def _parse(recent: dict):
        for form, filed, period, accno, doc in zip(
            recent.get("form",            []),
            recent.get("filingDate",      []),
            recent.get("reportDate",      []),
            recent.get("accessionNumber", []),
            recent.get("primaryDocument", []),
        ):
            if form not in TARGET_FORMS:
                continue
            if filed < cutoff:
                continue
            rows.append({
                "form_type":        form,
                "filed_date":       filed,
                "period_of_report": period or filed,
                "accession_no":     accno,
                "primary_document": doc,
            })

    _parse(data.get("filings", {}).get("recent", {}))

    for ef in data.get("filings", {}).get("files", [])[:5]:
        fname = ef.get("name", "")
        if not fname:
            continue
        r2 = sess.get(f"{sess.DATA_BASE}/submissions/{fname}")
        if r2:
            try:
                _parse(r2.json())
            except Exception:
                pass

    # Deduplicate
    seen, unique = set(), []
    for row in rows:
        if row["accession_no"] not in seen:
            seen.add(row["accession_no"])
            unique.append(row)

    print(f"    Found {len(unique)} target filings ({', '.join(sorted(TARGET_FORMS))})")
    return unique


# ─── DOCUMENT SELECTION ──────────────────────────────────────────────────────

def _is_xbrl_viewer_stub(fname: str) -> bool:
    """R1.htm … R99.htm are XBRL viewer stubs, not the real report."""
    return bool(re.match(r"^R\d+\.htm$", fname, re.IGNORECASE))


def _doc_score(doc: dict) -> int:
    """
    Score a document entry from the filing index.
    Higher = more likely to be the full narrative report.
    """
    fname = (doc.get("document") or "").lower()
    desc  = (doc.get("description") or "").lower()
    dtype = (doc.get("type") or "").lower()
    size  = doc.get("size") or 0

    if _is_xbrl_viewer_stub(fname):
        return -999
    if not fname.endswith(".htm"):
        return -1
    if any(p in fname for p in ["cover", "signature", "ex-", "exhibit", "r99", "r9"]):
        return -10

    score = 0
    # Strong signals for the main report
    if any(kw in desc for kw in ["10-k", "10-q", "annual report", "quarterly report",
                                  "form 10", "complete submission"]):
        score += 50
    if dtype in ("10-k", "10-q", "10-q/a", "10-k/a"):
        score += 40
    # Prefer larger files (more content)
    score += min(size // 50_000, 60)
    # Penalise very small files
    if size < MIN_HTML_BYTES:
        score -= 30

    return score


def _get_index_docs(cik: str, accno: str, sess: EDGARSession) -> list:
    clean = accno.replace("-", "")
    r = sess.get(f"{sess.ARCHIVES}/{int(cik)}/{clean}/{accno}-index.json")
    if not r:
        return []
    try:
        return r.json().get("documents", [])
    except Exception:
        return []


def resolve_best_html_url(
    cik: str, accno: str, primary_doc: str, sess: EDGARSession
) -> Optional[str]:
    """
    Find and return the URL of the best full narrative HTML document
    for a filing. Tries primary_document first, then ranks all .htm
    files in the filing index.
    """
    clean    = accno.replace("-", "")
    base_url = f"{sess.ARCHIVES}/{int(cik)}/{clean}"

    # Try declared primary document first
    if primary_doc and primary_doc.lower().endswith(".htm"):
        if not _is_xbrl_viewer_stub(primary_doc):
            url = f"{base_url}/{primary_doc}"
            r   = sess.get(url, timeout=120)
            if r and len(r.content) >= MIN_HTML_BYTES:
                return url

    # Fall back: rank all .htm files from the filing index
    docs = _get_index_docs(cik, accno, sess)
    if not docs:
        # If index fetch fails, try the primary doc anyway
        if primary_doc:
            return f"{base_url}/{primary_doc}"
        return None

    ranked = sorted(docs, key=_doc_score, reverse=True)

    for doc in ranked[:8]:
        fname = doc.get("document", "")
        if not fname or not fname.lower().endswith(".htm"):
            continue
        if _doc_score(doc) < 0:
            continue
        url = f"{base_url}/{fname}"
        r   = sess.get(url, timeout=120)
        if r and len(r.content) >= MIN_HTML_BYTES:
            mb = len(r.content) / 1_048_576
            if mb > MAX_HTML_MB:
                print(f"    [SKIP] Too large ({mb:.0f} MB): {fname}")
                continue
            return url

    return None


# ─── OUTPUT PATH ─────────────────────────────────────────────────────────────

def _safe(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]", "_", s)


def output_path(ticker: str, filing: dict, root: str = OUTPUT_DIR) -> Path:
    """
    filings/AAPL/2024/10-K_2024-09-28_0000320193-24-000123.html
    """
    period = filing["period_of_report"] or filing["filed_date"]
    year   = period[:4]
    folder = Path(root) / ticker.upper() / year
    folder.mkdir(parents=True, exist_ok=True)
    fname  = f"{_safe(filing['form_type'])}_{_safe(period)}_{_safe(filing['accession_no'])}.html"
    return folder / fname


# ─── DOWNLOAD ONE FILING ─────────────────────────────────────────────────────

def download_one(
    ticker: str,
    cik:    str,
    filing: dict,
    sess:   EDGARSession,
    root:   str  = OUTPUT_DIR,
    resume: bool = RESUME,
) -> Optional[Path]:
    """
    Download, clean, and save the HTML for one filing.
    Returns the saved Path on success, None on failure/skip.
    """
    dest = output_path(ticker, filing, root)

    # Resume: skip if file exists and is large enough
    if resume and dest.exists() and dest.stat().st_size >= MIN_HTML_BYTES:
        print(f"    [SKIP] Already on disk: {dest.name}")
        return dest

    url = resolve_best_html_url(cik, filing["accession_no"], filing["primary_document"], sess)
    if not url:
        print(f"    [WARN] Could not resolve HTML for {filing['accession_no']}")
        return None

    print(f"    [DL]  {filing['form_type']} | {filing['period_of_report']} → {dest.name}")
    r = sess.get(url, timeout=180)
    if not r:
        print(f"    [ERR] Download failed: {url[-70:]}")
        return None

    mb = len(r.content) / 1_048_576
    if mb > MAX_HTML_MB:
        print(f"    [SKIP] File too large ({mb:.0f} MB)")
        return None

    html = r.text

    # Clean iXBRL wrapper tags so financial content is fully readable
    if CLEAN_IXBRL and is_ixbrl(html):
        html = clean_ixbrl(html)
        print(f"    [iXBRL cleaned] {mb:.2f} MB → {len(html)/1_048_576:.2f} MB")

    dest.write_text(html, encoding="utf-8", errors="replace")
    print(f"    [OK]  Saved {len(html)/1_048_576:.2f} MB → {dest}")
    return dest


# ─── MAIN ORCHESTRATOR ────────────────────────────────────────────────────────

def download_all(
    tickers:    list,
    root:       str  = OUTPUT_DIR,
    years_back: int  = YEARS_BACK,
    resume:     bool = RESUME,
) -> dict:
    """
    Download all target filings for the given ticker list.
    Returns summary dict with counts.
    """
    sess          = EDGARSession()
    total_saved   = 0
    total_skipped = 0
    total_failed  = 0
    manifest      = []

    # Normalise input (accept plain strings or dicts)
    ticker_list = [
        t if isinstance(t, dict) else {"ticker": t, "name": t}
        for t in tickers
    ]

    print(f"\nSEC HTML Downloader v2.0")
    print(f"{'─'*60}")
    print(f"  Tickers     : {len(ticker_list)}")
    print(f"  Years back  : {years_back}  (cutoff: {(datetime.now()-timedelta(days=years_back*365)).strftime('%Y-%m-%d')})")
    print(f"  Forms       : {sorted(TARGET_FORMS)}")
    print(f"  Clean iXBRL : {CLEAN_IXBRL}  ← strips XBRL wrappers, keeps all financial text")
    print(f"  Resume      : {resume}")
    print(f"  Output root : {Path(root).resolve()}")
    print(f"{'─'*60}\n")

    for idx, company in enumerate(ticker_list, 1):
        ticker = company["ticker"]
        name   = company.get("name", ticker)
        print(f"[{idx}/{len(ticker_list)}] {ticker} — {name}")

        cik = resolve_cik(ticker, sess)
        if not cik:
            total_failed += 1
            manifest.append({"ticker": ticker, "saved": 0, "skipped": 0, "failed": 1, "note": "no CIK"})
            continue

        print(f"  CIK: {cik}")
        filings = get_target_filings(cik, sess, years_back=years_back)
        if not filings:
            manifest.append({"ticker": ticker, "saved": 0, "skipped": 0, "failed": 0, "note": "no filings"})
            continue

        saved = skipped = failed = 0
        for filing in filings:
            result = download_one(ticker, cik, filing, sess, root=root, resume=resume)
            dest   = output_path(ticker, filing, root)
            if result is None:
                if dest.exists():
                    skipped += 1    # was skipped by resume
                else:
                    failed  += 1
            else:
                saved += 1

        total_saved   += saved
        total_skipped += skipped
        total_failed  += failed
        manifest.append({
            "ticker":   ticker,
            "saved":    saved,
            "skipped":  skipped,
            "failed":   failed,
            "filings":  len(filings),
        })

        # Reset HTTP session every 20 companies
        if idx % 20 == 0:
            sess.reset()

    # ── Summary ──────────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print("DONE")
    print(f"{'═'*60}")
    print(f"  Tickers processed : {len(ticker_list)}")
    print(f"  Files saved       : {total_saved}   (new downloads, iXBRL cleaned)")
    print(f"  Files skipped     : {total_skipped}  (already on disk)")
    print(f"  Failures          : {total_failed}")
    print(f"  Output root       : {Path(root).resolve()}")

    # Write manifest
    mpath = Path(root) / "manifest.json"
    Path(root).mkdir(parents=True, exist_ok=True)
    with open(mpath, "w", encoding="utf-8") as f:
        json.dump({
            "generated_at":   datetime.utcnow().isoformat(),
            "years_back":     years_back,
            "target_forms":   sorted(TARGET_FORMS),
            "clean_ixbrl":    CLEAN_IXBRL,
            "total_saved":    total_saved,
            "total_skipped":  total_skipped,
            "total_failed":   total_failed,
            "tickers":        manifest,
        }, f, indent=2)
    print(f"  Manifest          : {mpath}\n")

    return {"saved": total_saved, "skipped": total_skipped, "failed": total_failed}


In [10]:


# ─── ENTRY POINT ──────────────────────────────────────────────────────────────

tickers = ALL_TICKERS

# ── Slice control ─────────────────────────────────────────────────────────────
# Uncomment ONE line to limit the run, or leave all commented for the full list:
tickers = tickers            # first 100
# tickers = tickers[:3]      # 3-company smoke-test
# tickers = tickers[:10]     # 10-company dev run
# tickers = tickers[:50]     # 50-company partial run
# tickers = tickers[100:200] # second batch of 100
# tickers = tickers[200:300] # third batch of 100
# ─────────────────────────────────────────────────────────────────────────────

download_all(tickers, root=OUTPUT_DIR, years_back=YEARS_BACK, resume=RESUME)


SEC HTML Downloader v2.0
────────────────────────────────────────────────────────────
  Tickers     : 582
  Years back  : 7  (cutoff: 2019-05-26)
  Forms       : ['10-K', '10-K/A', '10-Q', '10-Q/A']
  Clean iXBRL : True  ← strips XBRL wrappers, keeps all financial text
  Resume      : True
  Output root : /Users/fanilshakirov/conda/projects/practice_projects/01_finetuning/v21/01/filings
────────────────────────────────────────────────────────────

[1/582] AAPL — Apple Inc
  CIK: 0000320193
    Found 28 target filings (10-K, 10-K/A, 10-Q, 10-Q/A)
    [DL]  10-Q | 2026-03-28 → 10-Q_2026-03-28_0000320193-26-000013.html
    [iXBRL cleaned] 0.95 MB → 0.74 MB
    [OK]  Saved 0.74 MB → filings/AAPL/2026/10-Q_2026-03-28_0000320193-26-000013.html
    [DL]  10-Q | 2025-12-27 → 10-Q_2025-12-27_0000320193-26-000006.html
    [iXBRL cleaned] 0.73 MB → 0.57 MB
    [OK]  Saved 0.57 MB → filings/AAPL/2025/10-Q_2025-12-27_0000320193-26-000006.html
    [DL]  10-K | 2025-09-27 → 10-K_2025-09-27_000032019

KeyboardInterrupt: 